# Module 18: Training with Transformers


## 🏋️ Fine-Tuning a Transformer

In Module 14, we trained a custom NER model using a CNN (`tok2vec`).
If we want ultimate accuracy, we can replace the CNN with a Transformer and train it on our dataset.

This process is called **Fine-Tuning**. We take a model that already understands the English language perfectly (like BERT), and we just tweak its weights slightly so it learns our new `GADGET` entity.

### Step 1: The Configuration
When generating your `config.cfg` file, you must tell spaCy to optimize for accuracy rather than efficiency.

```bash
python -m spacy init config config.cfg --lang en --pipeline ner --optimize accuracy
```

If you open the generated `config.cfg`, you will notice the `tok2vec` block has been entirely replaced by a `transformer` block!


<br><br>

---

<br><br>


## ⚙️ Customizing the HuggingFace Model

The beauty of `spacy-transformers` is that you are not locked into a specific model.

Inside your `config.cfg`, you will see a setting like this:
```ini
[components.transformer.model]
@architectures = "spacy-transformers.TransformerModel.v3"
name = "roberta-base"
```

You can change `name` to **ANY text-classification model on HuggingFace!**
- Need a tiny, fast model? Change it to `"distilbert-base-uncased"`
- Need a multilingual model? Change it to `"xlm-roberta-base"`
- Need a medical model? Change it to `"emilyalsentzer/Bio_ClinicalBERT"`

When you run the `spacy train` command, spaCy will automatically download those exact weights from the HuggingFace Hub and wire them into your pipeline!


<br><br>

---

<br><br>


## 🥶 Freezing the Transformer

Transformers are massive (hundreds of millions of parameters). Training all of them takes massive amounts of VRAM (GPU memory).

If your GPU runs out of memory (OOM error), you can freeze the underlying transformer and *only* train the NER head that sits on top of it. This makes training much faster and uses far less memory.

In your `config.cfg`:
```ini
[training]
frozen_components = ["transformer"]
```


<br><br>

---

<br><br>


## 📉 Advanced GPU Techniques

If you are training on a consumer GPU (like an RTX 3060 with 12GB VRAM), you might still struggle to fit a Transformer batch into memory.

spaCy supports two critical MLOps techniques in the config file to help you:

1. **Mixed Precision (`use_amp`)**: By setting `accumulate_gradient = 1` and utilizing AMP, the GPU uses 16-bit math instead of 32-bit math. This cuts VRAM usage almost in half without hurting accuracy.
2. **Gradient Accumulation**: If your GPU can only handle a batch size of `2` without crashing, but you *want* a batch size of `8` for better learning, you can set `accumulate_gradient = 4`. spaCy will run 4 mini-batches of 2, add the math up quietly, and then do one massive weight update as if the batch was 8!


<br><br>

---

<br><br>


## 🎉 Summary of Part 6

You have now bridged the gap between spaCy's production pipelines and HuggingFace's cutting-edge deep learning research.

- You know how to use `spacy-transformers` to load massive models like RoBERTa.
- You understand the tradeoffs between CNNs (speed/efficiency) and Transformers (accuracy/context).
- You know how to edit the `config.cfg` file to inject any HuggingFace model into your pipeline, freeze its weights, and optimize GPU memory using gradient accumulation.

In the final section of our course, **Part 7: Project & Workflow Management**, we will learn how to wrap our code, models, and data into shareable **spaCy Projects**!
